# Föreläsning 5
## Introduktion till maskininlärning — regression

**Från data till en modell som kan göra förutsägelser**

# Idag

Efter träffen ska du kunna beskriva och använda arbetsflödet

**data → features/target → train/test → fit → predict → evaluate**

Vi börjar med **regression**: modellen ska förutsäga ett tal.

# Vad är maskininlärning?

I vanlig programmering skriver vi reglerna själva.

I maskininlärning visar vi i stället **exempel** och låter en modell hitta ett samband.

Exempel:

- bostadens egenskaper → pris
- temperatur → energianvändning
- motordata → bränsleförbrukning

# Två ord vi behöver

**Features (X)**  
Information modellen får använda.

**Target (y)**  
Det vi vill att modellen ska förutsäga.

> Vilka skulle vara features och target om vi vill förutsäga priset på en bostad?

# Ett litet bostadsdataset

Vi skapar ett syntetiskt dataset så att vi känner till ungefär hur datan har uppstått.

Priset påverkas av storlek, antal rum och ålder — plus lite slumpmässig variation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 140

housing = pd.DataFrame({
    "size_m2": rng.uniform(35, 160, n),
    "rooms": rng.integers(1, 6, n),
    "age_years": rng.integers(0, 80, n),
})

housing["price_ksek"] = (
    500
    + 38 * housing["size_m2"]
    + 160 * housing["rooms"]
    - 7 * housing["age_years"]
    + rng.normal(0, 350, n)
)

housing.head()

# Titta på datan först

Maskininlärning börjar fortfarande med dataanalys.

> Ser priset ut att hänga ihop med bostadens storlek?

In [ ]:
housing.plot.scatter(x="size_m2", y="price_ksek")
plt.xlabel("Storlek (m²)")
plt.ylabel("Pris (tusen kr)")
plt.title("Bostadspris och storlek")
plt.show()

# Första modellen: bara en feature

Vi låter modellen använda **storlek** för att förutsäga **pris**.

Scikit-learn förväntar sig oftast att `X` är en tabell och `y` är en serie.

In [ ]:
X = housing[["size_m2"]]
y = housing["price_ksek"]

print(X.head())
print()
print(y.head())

# Varför delar vi datan?

Om modellen tränas och examineras på samma data vet vi inte om den kan fungera på **nya observationer**.

Därför delar vi:

- **träning**: modellen får lära sig här
- **test**: hålls undan tills vi vill utvärdera modellen

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

print("Träning:", len(X_train))
print("Test:", len(X_test))

# Modellen

Vi börjar med **linjär regression**.

Den försöker hitta en rak linje:

**pris ≈ lutning × storlek + konstant**

I vårt gamla ML-material skrev vi själva ut modellens parametrar och uppdaterade dem. Här låter vi scikit-learn sköta själva träningen.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

# Vad lärde sig modellen?

`fit()` betyder: **anpassa modellens parametrar till träningsdatan**.

In [ ]:
print("Lutning:", model.coef_[0])
print("Konstant:", model.intercept_)

# Prediction

Nu får modellen data den inte såg under träningen.

Vi brukar skilja på:

- **ground truth / facit**: `y_test`
- **prediction / förutsägelse**: modellens svar

In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred
})
results.head()

# Är förutsägelserna rimliga?

Ett enkelt sätt är att jämföra facit och prediction visuellt.

En perfekt modell skulle ligga på diagonalen.

In [ ]:
plt.scatter(results["actual"], results["predicted"])
low = min(results.min())
high = max(results.max())
plt.plot([low, high], [low, high], "--")
plt.xlabel("Verkligt pris")
plt.ylabel("Förutsagt pris")
plt.title("Facit mot prediction")
plt.show()

# Ett felmått: MAE

**Mean Absolute Error** svarar ungefär på:

> Hur mycket missar modellen i genomsnitt?

För våra priser blir enheten fortfarande **tusen kronor**.

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.0f} tusen kr")

# Men är det bra?

Ett felmått behöver en jämförelse.

En mycket enkel **baseline** kan vara:

> Gissa alltid medelpriset i träningsdatan.

In [ ]:
baseline_pred = np.full(len(y_test), y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline MAE: {baseline_mae:.0f}")
print(f"Modellens MAE: {mae:.0f}")

# Fler features

Storlek är inte den enda information vi har.

> Tror ni modellen blir bättre om den också får antal rum och bostadens ålder?

In [ ]:
X = housing[["size_m2", "rooms", "age_years"]]
y = housing["price_ksek"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model_multi = LinearRegression()
model_multi.fit(X_train, y_train)
y_pred_multi = model_multi.predict(X_test)

mae_multi = mean_absolute_error(y_test, y_pred_multi)
print(f"MAE med tre features: {mae_multi:.0f} tusen kr")

# Samma arbetsflöde

Det viktiga är inte just `LinearRegression`.

Mönstret är:

1. välj `X` och `y`
2. dela train/test
3. skapa modell
4. `fit()`
5. `predict()`
6. utvärdera

Detta återkommer i nästan all scikit-learn.

# Kan en rak linje lära sig allt?

Nej.

I det tidigare regressionsmaterialet jämfördes också linjära och icke-linjära samband.

Vi skapar nu data där sambandet tydligt är böjt.

In [ ]:
x_curve = np.linspace(-4, 4, 100)
y_curve = x_curve**2 + rng.normal(0, 1.5, 100)

curve = pd.DataFrame({"x": x_curve, "y": y_curve})
curve.plot.scatter(x="x", y="y")
plt.title("Ett icke-linjärt samband")
plt.show()

In [ ]:
curve_model = LinearRegression()
curve_model.fit(curve[["x"]], curve["y"])
curve["prediction"] = curve_model.predict(curve[["x"]])

plt.scatter(curve["x"], curve["y"], label="data")
plt.plot(curve["x"], curve["prediction"], label="linjär modell")
plt.legend()
plt.show()

# En viktig ML-idé

En modell kan bara lära sig samband som dess **modellklass** kan representera.

En rak linje kan inte bli en U-formad kurva, hur länge vi än tränar den.

Detta är inte samma sak som att Pythonkoden är fel.

# Fråga innan vi avslutar

Vilket av följande är **inte** ett rimligt sätt att förbättra en modell?

A. Ge modellen relevanta features  
B. Samla mer representativ data  
C. Utvärdera på testdata och sedan träna om tills testresultatet blir perfekt  
D. Prova en modell som bättre passar typen av samband

# Idag

Vi har gått från en DataFrame till en tränad modell:

**Data → X/y → train/test → fit → predict → MAE → baseline**

Nästa gång byter vi från att förutsäga **tal** till att förutsäga **klasser** — och tittar närmare på hur modeller kan överanpassa sig.